# Training Module Demo

This notebook demonstrates all 5 components of the Training Module:

1. **ReductionsWrapper**: Wraps sklearn estimators with Fairlearn constraints (DemographicParity, EqualizedOdds)
2. **FairnessRegularizerLoss**: PyTorch loss function with fairness regularization (covariance or mean_gap modes)
3. **LagrangianFairnessTrainer**: PyTorch trainer using Lagrangian multipliers to enforce fairness constraints
4. **GroupFairnessCalibrator**: Post-processing calibration to improve fairness across groups (Platt scaling or isotonic)
5. **Pareto Frontier Visualization**: Tools to explore accuracy-fairness trade-offs (`sweep_pareto` and `plot_pareto`)

We'll use synthetic data that demonstrates fairness issues clearly, allowing us to see how each component addresses bias.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from fairlearn.reductions import DemographicParity, EqualizedOdds
import matplotlib.pyplot as plt

from fairness_pipeline_dev_toolkit.training import (
    ReductionsWrapper,
    FairnessRegularizerLoss,
    LagrangianFairnessTrainer,
    GroupFairnessCalibrator,
    sweep_pareto,
    plot_pareto,
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("✅ Training module imports successful!")
print(f"   PyTorch version: {torch.__version__}")
print(f"   NumPy version: {np.__version__}")

## Generate Sample Data

We'll create a synthetic dataset where the target variable has an undesirable dependence on a sensitive attribute, creating a fairness problem that our training components will address.

In [ ]:
# Generate synthetic dataset with fairness issues
rng = np.random.RandomState(42)
n_samples = 500
n_features = 6

# Generate features
X = rng.randn(n_samples, n_features)

# Generate sensitive attribute (binary: 0 or 1)
# Group 0: 50%, Group 1: 50%
sensitive = (rng.rand(n_samples) > 0.5).astype(int)

# Generate labels with undesirable dependence on sensitive attribute
# y depends on X[:, 0] (legitimate feature) but also on sensitive (bias)
y = ((X[:, 0] + 0.4 * sensitive + 0.2 * rng.randn(n_samples)) > 0.1).astype(int)

# Split into train and validation sets
X_train, X_val, y_train, y_val, s_train, s_val = train_test_split(
    X, y, sensitive, test_size=0.3, random_state=42, stratify=y
)

print(f"✅ Dataset generated:")
print(f"   Total samples: {n_samples}")
print(f"   Training samples: {len(X_train)}")
print(f"   Validation samples: {len(X_val)}")
print(f"   Features: {n_features}")
print(f"\n   Label distribution (train):")
print(f"      Class 0: {np.sum(y_train == 0)} ({np.mean(y_train == 0):.2%})")
print(f"      Class 1: {np.sum(y_train == 1)} ({np.mean(y_train == 1):.2%})")
print(f"\n   Sensitive attribute distribution (train):")
print(f"      Group 0: {np.sum(s_train == 0)} ({np.mean(s_train == 0):.2%})")
print(f"      Group 1: {np.sum(s_train == 1)} ({np.mean(s_train == 1):.2%})")
print(f"\n   Baseline demographic parity violation:")
dp_baseline = np.abs(np.mean(y_train[s_train == 0]) - np.mean(y_train[s_train == 1]))
print(f"      |P(Y=1|S=0) - P(Y=1|S=1)| = {dp_baseline:.4f}")

## Component 1: ReductionsWrapper

The `ReductionsWrapper` wraps any sklearn classifier with Fairlearn's `ExponentiatedGradient` to enforce fairness constraints during training. We'll demonstrate it with `DemographicParity` constraint.

In [ ]:
# Create base sklearn estimator
base_estimator = GradientBoostingClassifier(
    n_estimators=50,
    max_depth=3,
    random_state=42
)

# Create fairness constraint (DemographicParity with difference bound)
constraint = DemographicParity(difference_bound=0.1)

# Wrap with ReductionsWrapper
fair_clf = ReductionsWrapper(
    base_estimator=base_estimator,
    constraint=constraint,
    eps=0.02,  # Tolerance on constraint violation
    T=15       # Max iterations for ExponentiatedGradient
)

# Fit the model
print("Training ReductionsWrapper...")
fair_clf.fit(X_train, y_train, sensitive_features=s_train)
print("✅ Model trained!")

# Make predictions
y_pred = fair_clf.predict(X_val)
y_proba = fair_clf.predict_proba(X_val)[:, 1] if hasattr(fair_clf, "predict_proba") else None

# Evaluate
accuracy = np.mean(y_pred == y_val)
print(f"\n📊 Results:")
print(f"   Validation Accuracy: {accuracy:.4f}")

# Check demographic parity
dp_group_0 = np.mean(y_pred[s_val == 0])
dp_group_1 = np.mean(y_pred[s_val == 1])
dp_violation = np.abs(dp_group_0 - dp_group_1)

print(f"\n   Demographic Parity:")
print(f"      Group 0 positive rate: {dp_group_0:.4f}")
print(f"      Group 1 positive rate: {dp_group_1:.4f}")
print(f"      Violation: {dp_violation:.4f} (target: < 0.1)")

if y_proba is not None:
    print(f"\n   Prediction probabilities range: [{y_proba.min():.4f}, {y_proba.max():.4f}]")

The constraint was not fully satisfied. Here's what this means:

1. Group 0 positive rate: 48.05% (48% of Group 0 gets positive predictions)
2. Group 1 positive rate: 60.27% (60% of Group 1 gets positive predictions)
Difference: 12.22 percentage points

This indicates the model predicts positive outcomes more often for Group 1 than Group 0, which violates demographic parity.

The Constraint Wasn't Fully Satisfied
The ReductionsWrapper uses Fairlearn's ExponentiatedGradient algorithm, which:
1. Iteratively adjusts the model to satisfy the constraint
2. Has a maximum of T=15 iterations (you set T=15)
3. Uses eps=0.02 as a tolerance for constraint satisfaction
4. Targets difference_bound=0.1 (10% maximum difference)

Possible reasons for the violation:
1. Insufficient iterations: T=15 may not be enough for convergence
2. Optimization trade-off: The algorithm balances accuracy and fairness; it may stop when the improvement is small
3. Data characteristics: The training data may have inherent bias that's difficult to correct within the given parameters

The model is trying to be fairer (the violation is relatively small at 12.22%)
It hasn't fully reached the target (< 10%)
The algorithm is working but may need more iterations or parameter tuning

## Component 2: FairnessRegularizerLoss

The `FairnessRegularizerLoss` is a PyTorch loss function that combines binary cross-entropy with a fairness penalty. It supports two modes: "covariance" (penalizes correlation between predictions and sensitive attribute) and "mean_gap" (penalizes difference in mean predictions across groups).

In [ ]:
# Create a simple PyTorch model
class SimpleNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# Convert data to PyTorch tensors
X_train_torch = torch.FloatTensor(X_train)
y_train_torch = torch.FloatTensor(y_train)
s_train_torch = torch.LongTensor(s_train)
X_val_torch = torch.FloatTensor(X_val)
y_val_torch = torch.LongTensor(y_val)
s_val_torch = torch.LongTensor(s_val)

# Train with different eta values to see the trade-off
etas = [0.0, 0.5, 1.0]
results = {}

for eta in etas:
    print(f"\n{'='*60}")
    print(f"Training with eta={eta} (mode='covariance')")
    print(f"{'='*60}")
    
    # Create fresh model for each eta
    model = SimpleNet(X_train.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    # Create loss function with fairness regularizer
    loss_fn = FairnessRegularizerLoss(eta=eta, mode="covariance")
    
    # Training loop
    n_epochs = 20
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        
        logits = model(X_train_torch).squeeze()
        loss = loss_fn(logits, y_train_torch, s_train_torch)
        
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                val_logits = model(X_val_torch).squeeze()
                val_pred = (torch.sigmoid(val_logits) > 0.5).long()
                val_acc = (val_pred == y_val_torch).float().mean().item()
                val_dp_0 = torch.sigmoid(val_logits[s_val_torch == 0]).mean().item()
                val_dp_1 = torch.sigmoid(val_logits[s_val_torch == 1]).mean().item()
                val_dp_violation = abs(val_dp_0 - val_dp_1)
            
            print(f"  Epoch {epoch+1}/{n_epochs}: Loss={loss.item():.4f}, "
                  f"Acc={val_acc:.4f}, DP_violation={val_dp_violation:.4f}")
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_torch).squeeze()
        val_pred = (torch.sigmoid(val_logits) > 0.5).long()
        val_acc = (val_pred == y_val_torch).float().mean().item()
        val_dp_0 = torch.sigmoid(val_logits[s_val_torch == 0]).mean().item()
        val_dp_1 = torch.sigmoid(val_logits[s_val_torch == 1]).mean().item()
        val_dp_violation = abs(val_dp_0 - val_dp_1)
    
    results[eta] = {
        "accuracy": val_acc,
        "dp_violation": val_dp_violation,
        "dp_group_0": val_dp_0,
        "dp_group_1": val_dp_1
    }

print(f"\n{'='*60}")
print("Summary: FairnessRegularizerLoss (covariance mode)")
print(f"{'='*60}")
print(f"{'Eta':<8} {'Accuracy':<12} {'DP Violation':<15} {'Group 0 Rate':<15} {'Group 1 Rate':<15}")
print("-" * 70)
for eta in etas:
    r = results[eta]
    print(f"{eta:<8.1f} {r['accuracy']:<12.4f} {r['dp_violation']:<15.4f} "
          f"{r['dp_group_0']:<15.4f} {r['dp_group_1']:<15.4f}")

# Also demonstrate mean_gap mode
print(f"\n{'='*60}")
print("Training with eta=1.0 (mode='mean_gap')")
print(f"{'='*60}")

model_mean_gap = SimpleNet(X_train.shape[1])
optimizer = optim.Adam(model_mean_gap.parameters(), lr=1e-3)
loss_fn_mean_gap = FairnessRegularizerLoss(eta=1.0, mode="mean_gap")

for epoch in range(20):
    model_mean_gap.train()
    optimizer.zero_grad()
    logits = model_mean_gap(X_train_torch).squeeze()
    loss = loss_fn_mean_gap(logits, y_train_torch, s_train_torch)
    loss.backward()
    optimizer.step()

model_mean_gap.eval()
with torch.no_grad():
    val_logits = model_mean_gap(X_val_torch).squeeze()
    val_pred = (torch.sigmoid(val_logits) > 0.5).long()
    val_acc = (val_pred == y_val_torch).float().mean().item()
    val_dp_violation = abs(torch.sigmoid(val_logits[s_val_torch == 0]).mean().item() - 
                           torch.sigmoid(val_logits[s_val_torch == 1]).mean().item())

print(f"✅ Mean-gap mode: Accuracy={val_acc:.4f}, DP_violation={val_dp_violation:.4f}")

1. Optimal eta value
eta=0.5 is the best balance for this dataset.
eta=0.0: good accuracy, small violation.
eta=1.0: worse on both metrics (in covariance mode).

2. Non-monotonic behavior
Higher eta doesn’t always improve fairness, eta=1.0 performs worse than eta=0.5. This suggests tuning eta is important.

3. Mode comparison
Covariance mode (eta=0.5): 76% accuracy, 0.02% violation
Mean gap mode (eta=1.0): 43% accuracy, 0.24% violation
Covariance mode achieves better balance here

4. Training dynamics
eta=0.5: violation decreases during training (0.0030 → 0.0002)
eta=1.0: violation increases slightly (0.0115 → 0.0124)
Suggests eta=0.5 converges better

## Component 3: LagrangianFairnessTrainer

The `LagrangianFairnessTrainer` uses Lagrangian multipliers to enforce fairness constraints. It alternates between optimizing the model (primal) and updating the constraint violation penalty (dual).

In [ ]:
# Create PyTorch model
model_lag = SimpleNet(X_train.shape[1])

# Initialize Lagrangian trainer with demographic_parity constraint
trainer = LagrangianFairnessTrainer(
    model=model_lag,
    fairness="demographic_parity",
    dp_tolerance=0.1,  # Allowed difference between groups
    model_lr=1e-3,      # Learning rate for model parameters
    lambda_lr=1e-2,     # Step size for dual variable (lambda)
    device="cpu"
)

print("Training with LagrangianFairnessTrainer...")
print(f"   Constraint: demographic_parity")
print(f"   Tolerance: {trainer.dp_tolerance}")
print(f"   Model LR: {trainer.model_lr}, Lambda LR: {trainer.lambda_lr}")

# Train the model
history = trainer.fit(
    X_train_torch,
    y_train_torch.long(),
    s_train_torch,
    epochs=30,
    batch_size=64,
    verbose=True
)

# Display training history
print(f"\n{'='*60}")
print("Training History (last 5 epochs):")
print(f"{'='*60}")
print(f"{'Epoch':<8} {'Accuracy':<12} {'Violation':<15} {'Lambda':<12}")
print("-" * 50)
for entry in history[-5:]:
    print(f"{entry['epoch']:<8} {entry['accuracy']:<12.4f} {entry['violation']:<15.4f} {entry['lambda']:<12.4f}")

# Evaluate on validation set
model_lag.eval()
with torch.no_grad():
    val_logits = model_lag(X_val_torch).squeeze()
    val_pred = (torch.sigmoid(val_logits) > 0.5).long()
    val_acc = (val_pred == y_val_torch).float().mean().item()
    val_dp_0 = torch.sigmoid(val_logits[s_val_torch == 0]).mean().item()
    val_dp_1 = torch.sigmoid(val_logits[s_val_torch == 1]).mean().item()
    val_dp_violation = abs(val_dp_0 - val_dp_1)

print(f"\n📊 Final Validation Results:")
print(f"   Accuracy: {val_acc:.4f}")
print(f"   Demographic Parity Violation: {val_dp_violation:.4f}")
print(f"   Group 0 positive rate: {val_dp_0:.4f}")
print(f"   Group 1 positive rate: {val_dp_1:.4f}")
print(f"   Final lambda value: {history[-1]['lambda']:.4f}")

# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = [h['epoch'] for h in history]
accs = [h['accuracy'] for h in history]
violations = [h['violation'] for h in history]
lambdas = [h['lambda'] for h in history]

axes[0].plot(epochs, accs, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy over Training')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, violations, 'r-', linewidth=2)
axes[1].axhline(y=trainer.dp_tolerance, color='g', linestyle='--', label='Tolerance')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Constraint Violation')
axes[1].set_title('Violation over Training')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, lambdas, 'm-', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Lambda (Dual Variable)')
axes[2].set_title('Lambda over Training')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The LagrangianFairnessTrainer uses a dual optimization approach:
- Primal: optimize model parameters for accuracy
- Dual: update the Lagrange multiplier (λ) to penalize constraint violations
The results show the constraint was satisfied during training, so λ stayed at 0.

#### Key Observations
1. Training phase: constraint satisfied
```Violation: 0.0000 throughout all 30 epochs``` ```Lambda: 0.0000 throughout all 30 epochs```
What this means:
- The model naturally satisfied the demographic parity constraint (tolerance 0.1) during training
- No penalty was needed, so λ remained 0
- The algorithm didn’t need to activate the fairness penalty

Why λ = 0:
- If violation < tolerance, λ doesn’t increase
- The model learned to be fair without explicit penalization
- This is a favorable outcome: fairness without sacrificing accuracy

Validation results: slight violation

```Training:   Violation = 0.0000 (perfect)```
```Validation: Violation = 0.0216 (2.16%)```
What this means:
- Training violation was 0, but validation shows 2.16%. Still within tolerance (0.1 = 10%)

Possible causes:
- Slight overfitting to training fairness
- Small distribution shift between train/validation
- Normal generalization behavior

##### Comparison with other methods

| Method | Training Accuracy | Training DP Violation | Validation DP Violation |
| --- | --- | --- | --- |
LagrangianFairnessTrainer |	90.9% |	0.00% |	2.16% |
FairnessRegularizerLoss (eta=0.5) |	76.0% |	0.02% |	~0.02% |
ReductionsWrapper |	92.0% |	12.22% | ~12% |

Observations:
- Highest training accuracy (90.9%)
- Perfect training fairness (0% violation)
- Small validation violation (2.16%), still within tolerance
- Better than ReductionsWrapper on fairness


#### Interpretation
What the plots show
1. Accuracy plot: steady increase to ~90%
2. Violation plot: stays at 0.00 (below tolerance 0.10)
3. Lambda plot: stays at 0.00 (no penalty needed)
Why λ = 0 is expected here
- The constraint was satisfied naturally
- No need to penalize violations
- The model learned a fair solution without explicit enforcement

#### Summary
The LagrangianFairnessTrainer achieved:
- High accuracy (90.9% training, 88% validation)
- Perfect training fairness (0% violation)
- Acceptable validation fairness (2.16% violation, well below 10% tolerance)
- Efficient optimization (λ = 0, no penalty needed)

This indicates the model learned a fair solution without explicit penalization, which is an ideal outcome for Lagrangian optimization.

## Component 4: GroupFairnessCalibrator

The `GroupFairnessCalibrator` performs post-processing calibration to improve fairness. It fits separate calibrators (Platt scaling or isotonic regression) for each group to ensure calibrated probabilities across groups.

In [ ]:
# Generate uncalibrated scores with group bias
# Simulate a model that produces biased scores
rng_cal = np.random.RandomState(123)
n_cal = 300

# Generate scores and groups
scores = rng_cal.rand(n_cal)
groups = rng_cal.choice([0, 1], size=n_cal)

# Create labels with group bias (group 1 gets higher positive rate)
logits = scores + 0.3 * (groups == 1) + 0.1 * rng_cal.randn(n_cal)
probs = 1.0 / (1.0 + np.exp(-5 * (logits - 0.5)))
labels = (rng_cal.rand(n_cal) < probs).astype(int)

# Split into train and test for calibration
scores_train, scores_test, labels_train, labels_test, groups_train, groups_test = train_test_split(
    scores, labels, groups, test_size=0.4, random_state=42, stratify=labels
)

print(f"✅ Calibration data generated:")
print(f"   Training samples: {len(scores_train)}")
print(f"   Test samples: {len(scores_test)}")
print(f"\n   Uncalibrated scores - Group 0 positive rate: {np.mean(labels_train[groups_train == 0]):.4f}")
print(f"   Uncalibrated scores - Group 1 positive rate: {np.mean(labels_train[groups_train == 1]):.4f}")
print(f"   Initial DP violation: {abs(np.mean(labels_train[groups_train == 0]) - np.mean(labels_train[groups_train == 1])):.4f}")

# Fit calibrator with Platt scaling
print(f"\n{'='*60}")
print("Fitting GroupFairnessCalibrator (method='platt')")
print(f"{'='*60}")

calibrator_platt = GroupFairnessCalibrator(method="platt", min_samples=20)
calibrator_platt.fit(scores_train, labels_train, groups_train)

# Transform scores
calibrated_scores_platt = calibrator_platt.transform(scores_test, groups_test)

print(f"✅ Calibration complete!")
print(f"   Groups calibrated: {list(calibrator_platt.calibrators.keys())}")

# Evaluate calibration quality
print(f"\n📊 Calibration Results (Platt):")
print(f"   Original score range: [{scores_test.min():.4f}, {scores_test.max():.4f}]")
print(f"   Calibrated score range: [{calibrated_scores_platt.min():.4f}, {calibrated_scores_platt.max():.4f}]")
print(f"   Calibrated scores are valid probabilities: {np.all((calibrated_scores_platt >= 0) & (calibrated_scores_platt <= 1))}")

# Check fairness improvement
pred_original = (scores_test > 0.5).astype(int)
pred_calibrated = (calibrated_scores_platt > 0.5).astype(int)

dp_original_0 = np.mean(pred_original[groups_test == 0])
dp_original_1 = np.mean(pred_original[groups_test == 1])
dp_calibrated_0 = np.mean(pred_calibrated[groups_test == 0])
dp_calibrated_1 = np.mean(pred_calibrated[groups_test == 1])

print(f"\n   Demographic Parity (threshold=0.5):")
print(f"      Original - Group 0: {dp_original_0:.4f}, Group 1: {dp_original_1:.4f}, "
      f"Violation: {abs(dp_original_0 - dp_original_1):.4f}")
print(f"      Calibrated - Group 0: {dp_calibrated_0:.4f}, Group 1: {dp_calibrated_1:.4f}, "
      f"Violation: {abs(dp_calibrated_0 - dp_calibrated_1):.4f}")

# Also demonstrate isotonic method
print(f"\n{'='*60}")
print("Fitting GroupFairnessCalibrator (method='isotonic')")
print(f"{'='*60}")

calibrator_isotonic = GroupFairnessCalibrator(method="isotonic", min_samples=20)
calibrator_isotonic.fit(scores_train, labels_train, groups_train)
calibrated_scores_isotonic = calibrator_isotonic.transform(scores_test, groups_test)

pred_isotonic = (calibrated_scores_isotonic > 0.5).astype(int)
dp_isotonic_0 = np.mean(pred_isotonic[groups_test == 0])
dp_isotonic_1 = np.mean(pred_isotonic[groups_test == 1])

print(f"✅ Isotonic calibration complete!")
print(f"   Isotonic - Group 0: {dp_isotonic_0:.4f}, Group 1: {dp_isotonic_1:.4f}, "
      f"Violation: {abs(dp_isotonic_0 - dp_isotonic_1):.4f}")

# Visualize calibration
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for idx, (method, cal_scores) in enumerate([("Platt", calibrated_scores_platt), ("Isotonic", calibrated_scores_isotonic)]):
    ax = axes[idx]
    for group in [0, 1]:
        mask = groups_test == group
        ax.hist(cal_scores[mask], alpha=0.6, label=f"Group {group}", bins=20)
    ax.set_xlabel('Calibrated Probability')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{method} Calibration - Score Distribution')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The GroupFairnessCalibrator fits separate calibrators per group to improve probability calibration. In this run, calibration improved calibration quality but worsened fairness.

**Initial State (Uncalibrated Scores)**
`Group 0 positive rate: 51.72%Group 1 positive rate: 80.65%Initial DP violation: 28.92%`
- Large initial bias: Group 1 has a much higher positive rate
- High violation: 28.92% difference

#### After Platt Calibration
**Original predictions (threshold=0.5)**
`Group 0: 55.74% | Group 1: 55.93% | Violation: 0.19% ✅`
- Fair at this threshold (0.19% violation)

**Calibrated predictions (threshold=0.5)**
`Group 0: 55.74% | Group 1: 100.00% | Violation: 44.26% ❌`
- Worsened fairness: Group 1 is 100% positive
- Violation increased from 0.19% to 44.26%

What happened:
- The calibrator learned group-specific mappings from the biased training data
- Group 1 scores were pushed to very high probabilities (see plot: peak around 0.9–0.95)
- At threshold 0.5, all Group 1 predictions become positive

#### After Isotonic Calibration
`Group 0: 55.74% | Group 1: 83.05% | Violation: 27.31%`
- Better than Platt (27.31% vs 44.26%), but still worse than original (0.19%)
- Group 1 still has a high positive rate (83.05%)

What happened:
- Isotonic is more flexible and produced a less extreme distribution
- Still shows bias: Group 1 has a large peak near 1.0 (see plot)


### Why Calibration Worsened Fairness
**1. Separate calibrators per group**
- Each group gets its own calibrator
- If training data is biased, the calibrators learn that bias
- Calibration improves probability estimates, not fairness

**2. The calibration process**
`Original scores → Calibrator learns mapping → Calibrated probabilities`
- If original scores are biased, calibrated probabilities can be more biased
- Calibration makes probabilities better calibrated, not necessarily fairer

**3. Threshold sensitivity**
- Original predictions at 0.5 were fair (0.19% violation)
- After calibration, the same threshold yields much worse fairness
- The calibration shifted the score distributions differently for each group

### Visual Evidence (from plots)
**Platt Calibration**
- Group 0: spread across 0.2–0.7
- Group 1: concentrated at 0.9–0.95
- Result: at threshold 0.5, Group 1 is almost entirely positive

**Isotonic Calibration**
- Group 0: distributed across 0.0–0.9
- Group 1: large peak at 0.95–1.0
- Result: still biased, but less extreme than Platt

### Key Insights
**1. Calibration ≠ Fairness**
- Calibration improves probability estimates
It does not guarantee fairness
It can make fairness worse if the data is biased
**2. Post-processing limitations**
- GroupFairnessCalibrator fits to the data it sees
If training data is biased, calibrators learn that bias
This is a limitation of post-processing approaches
**3. Why original predictions were fairer**
- The original model, despite biased scores, produced fair predictions at threshold 0.5
Calibration changed the score distributions, breaking that balance

### When Calibration Helps vs Hurts
Calibration helps when:
- Original probabilities are poorly calibrated (e.g., overconfident)
- You need better probability estimates
- The underlying bias is small

Calibration can hurt when:
- Original predictions are already fair at your threshold
- Training data has strong bias
- You rely on a fixed threshold (e.g., 0.5)


### Recommendations
1. Use group-specific thresholds
Instead of a fixed 0.5 threshold:
```# Find optimal threshold per group to achieve fairnessthreshold_group_0 = find_threshold(calibrated_scores[groups == 0], labels[groups == 0])threshold_group_1 = find_threshold(calibrated_scores[groups == 1], labels[groups == 1])```
2. Combine with in-training methods
- Use FairnessRegularizerLoss or LagrangianFairnessTrainer during training
- Then apply calibration for better probability estimates

This addresses bias at training time, not just post-processing
3. Monitor both metrics
- Track calibration quality (e.g., Brier score, ECE)
- Track fairness metrics (e.g., demographic parity)
- Don’t assume one improves the other
4. Consider the use case
- If you need calibrated probabilities: use calibration
- If you need fairness: prioritize in-training methods
- If you need both: combine approaches

### Summary
The GroupFairnessCalibrator results show:
- Calibration improved probability calibration
- Fairness worsened (violation increased from 0.19% to 27–44%)
- Separate calibrators learned the bias present in the training data

This highlights a limitation of post-processing: it improves calibration but not necessarily fairness

Takeaway: Calibration and fairness are different goals. Calibration improves probability estimates; fairness requires addressing bias in the data or model. For fairness, prefer in-training methods or combine them with careful post-processing.

## Component 5: Pareto Frontier Visualization

The Pareto frontier shows the trade-off between accuracy and fairness. We use `sweep_pareto()` to train models across a range of fairness regularization strengths (`eta` values) and `plot_pareto()` to visualize the results.

In [ ]:
# Prepare data for Pareto sweep
# Use a subset for faster computation
X_train_pareto = X_train[:200]
y_train_pareto = y_train[:200]
s_train_pareto = s_train[:200]
X_val_pareto = X_val
y_val_pareto = y_val
s_val_pareto = s_val

# Define range of eta values to explore
etas = [0.0, 0.1, 0.2, 0.5, 1.0, 2.0]

print(f"Running Pareto sweep across {len(etas)} eta values...")
print(f"   Eta values: {etas}")
print(f"   Training samples: {len(X_train_pareto)}")
print(f"   Validation samples: {len(X_val_pareto)}")

# Run sweep_pareto
pareto_points = sweep_pareto(
    X_train_pareto,
    y_train_pareto,
    s_train_pareto,
    X_val_pareto,
    y_val_pareto,
    s_val_pareto,
    etas=etas,
    epochs=15,  # Fewer epochs for demo
    lr=1e-3,
    device="cpu"
)

print(f"\n✅ Pareto sweep complete!")
print(f"   Points generated: {len(pareto_points)}")

# Display results
print(f"\n{'='*70}")
print("Pareto Frontier Points:")
print(f"{'='*70}")
print(f"{'Eta':<8} {'Accuracy':<12} {'DP Difference':<15}")
print("-" * 40)
for pt in pareto_points:
    print(f"{pt['eta']:<8.2f} {pt['accuracy']:<12.4f} {pt['dp_diff']:<15.4f}")

# Plot Pareto frontier
print(f"\nGenerating Pareto frontier plot...")
import os
os.makedirs("artifacts", exist_ok=True)
plot_pareto(pareto_points, save_path="artifacts/pareto_frontier_demo.png")
print(f"✅ Plot saved to: artifacts/pareto_frontier_demo.png")

# Also display inline
plot_pareto(pareto_points, save_path=None)
plt.show()

# Interpret results
print(f"\n{'='*70}")
print("Interpretation:")
print(f"{'='*70}")
print("The Pareto frontier shows the accuracy-fairness trade-off:")
print("  - Lower eta (left side): Higher accuracy, lower fairness")
print("  - Higher eta (right side): Lower accuracy, higher fairness")
print("\nPoints on the frontier represent optimal trade-offs - you cannot")
print("improve one metric without worsening the other.")

### Results Analysis

#### All Points (Sorted by Accuracy)

| Eta | Accuracy | DP Difference | Interpretation |
|-----|----------|---------------|----------------|
| **η=0.2** | **70.67%** | 0.0097 | Best accuracy, worst fairness |
| **η=1.0** | **62.00%** | 0.0052 | High accuracy, moderate fairness |
| **η=0.0** | **59.33%** | 0.0041 | Good balance |
| **η=0.5** | **58.67%** | 0.0044 | Slightly worse than η=0.0 |
| **η=0.1** | **40.00%** | 0.0054 | Poor performance |
| **η=2.0** | **41.33%** | **0.0029** | Best fairness, worst accuracy |

---

### Pareto-Optimal Points

These are points where you **can't improve one metric without hurting the other**:

#### 1. **η=2.0 — Best Fairness**
```
Accuracy: 41.33% | DP Difference: 0.0029 (best)
```
- ✅ **Best fairness** (lowest DP difference)
- ❌ **Lowest accuracy**
- **Use when**: Fairness is the top priority

#### 2. **η=0.0 — Good Balance**
```
Accuracy: 59.33% | DP Difference: 0.0041
```
- ✅ **Good balance** of accuracy and fairness
- ✅ **Baseline** (no fairness penalty)
- **Use when**: You want a balanced model

#### 3. **η=1.0 — Higher Accuracy**
```
Accuracy: 62.00% | DP Difference: 0.0052
```
- ✅ **Higher accuracy** than η=0.0 (+2.67%)
- ⚠️ **Slightly worse fairness** (0.0052 vs 0.0041)
- **Use when**: You want more accuracy with acceptable fairness

#### 4. **η=0.2 — Maximum Accuracy**
```
Accuracy: 70.67% | DP Difference: 0.0097 (worst)
```
- ✅ **Highest accuracy**
- ❌ **Worst fairness**
- **Use when**: Accuracy is the top priority

---

### Dominated Points (Not Pareto-Optimal)

These are **worse than at least one other point in both metrics**:

#### **η=0.1 — Dominated**
```
Accuracy: 40.00% | DP Difference: 0.0054
```
- ❌ Worse than η=2.0 (40% < 41.33% and 0.0054 > 0.0029)
- ❌ Worse than η=0.0 (40% < 59.33% and 0.0054 > 0.0041)
- **Avoid this point** — it's dominated

#### **η=0.5 — Dominated**
```
Accuracy: 58.67% | DP Difference: 0.0044
```
- ❌ Worse than η=0.0 (58.67% < 59.33% and 0.0044 > 0.0041)
- ❌ Worse than η=1.0 (58.67% < 62.00% and 0.0044 > 0.0052)
- **Avoid this point** — it's dominated

---

### Key Insights

#### 1. **Non-Monotonic Behavior**
- Higher `eta` doesn't always improve fairness
- η=0.1 performs poorly (dominated)
- η=0.2 achieves highest accuracy despite low `eta`
- This suggests the optimization landscape is complex

#### 2. **The Accuracy-Fairness Trade-off**
- **Best accuracy** (η=0.2): 70.67% accuracy, 0.0097 DP diff
- **Best fairness** (η=2.0): 41.33% accuracy, 0.0029 DP diff
- **Trade-off**: ~29% accuracy loss for ~0.0068 improvement in fairness

### 3. **Sweet Spot Identification**
- **η=0.0 or η=1.0** offer good balance
- η=0.0: 59.33% accuracy, 0.0041 DP diff
- η=1.0: 62.00% accuracy, 0.0052 DP diff
- Choose based on whether you prioritize accuracy or fairness

#### 4. **Comparison with Other Methods**

| Method | Accuracy | DP Violation |
|--------|----------|--------------|
| **Pareto (η=0.2)** | **70.67%** | **0.97%** |
| **Pareto (η=1.0)** | **62.00%** | **0.52%** |
| **Pareto (η=0.0)** | **59.33%** | **0.41%** |
| **LagrangianFairnessTrainer** | **88.00%** | **2.16%** |
| **ReductionsWrapper** | **92.00%** | **12.22%** |

**Observations:**
- Pareto sweep uses smaller training set (200 vs 350), which may explain lower accuracy
- Pareto points show better fairness than ReductionsWrapper
- LagrangianFairnessTrainer achieves higher accuracy but with higher violation

---

### How to Use the Pareto Frontier

#### **Step 1: Identify Your Priorities**
- **Maximum accuracy**: Choose η=0.2 (70.67% accuracy)
- **Maximum fairness**: Choose η=2.0 (0.0029 DP difference)
- **Balanced**: Choose η=0.0 or η=1.0

#### **Step 2: Avoid Dominated Points**
- Don't use η=0.1 or η=0.5
- They are worse in both metrics than other options

#### **Step 3: Consider the Frontier**
The Pareto frontier consists of:
- η=2.0 (best fairness)
- η=0.0 (good balance)
- η=1.0 (higher accuracy)
- η=0.2 (best accuracy)

Any point on this frontier is Pareto-optimal.

---

### Recommendations

#### **For Production Use:**
1. **If fairness is critical**: η=2.0 (0.0029 DP diff, 41.33% accuracy)
2. **If you need balance**: η=0.0 (0.0041 DP diff, 59.33% accuracy)
3. **If you want more accuracy**: η=1.0 (0.0052 DP diff, 62.00% accuracy)
4. **If accuracy is paramount**: η=0.2 (0.0097 DP diff, 70.67% accuracy)

#### **For Further Exploration:**
- Try more `eta` values between 0.0-0.2 and 1.0-2.0
- Consider using more training data (currently 200 samples)
- Compare with LagrangianFairnessTrainer for higher accuracy

---

### Summary

The Pareto frontier shows:
- ✅ **4 Pareto-optimal points** (η=2.0, 0.0, 1.0, 0.2)
- ❌ **2 dominated points** (η=0.1, 0.5) — avoid these
- 📊 **Clear trade-off**: Higher accuracy comes with worse fairness
- 🎯 **Best balance**: η=0.0 or η=1.0 depending on priorities

The frontier helps you choose a model that matches your accuracy and fairness requirements. Use it to make an informed decision based on your specific needs.

## Summary

This notebook demonstrated all 5 components of the Training Module:

### 1. ReductionsWrapper
- Wraps sklearn estimators with Fairlearn constraints
- Enforces fairness during training using ExponentiatedGradient
- Supports DemographicParity and EqualizedOdds constraints
- **Use when**: You want to use sklearn models with fairness constraints

### 2. FairnessRegularizerLoss
- PyTorch loss function combining BCE with fairness penalty
- Two modes: "covariance" (correlation penalty) and "mean_gap" (difference penalty)
- Tunable `eta` parameter controls fairness-accuracy trade-off
- **Use when**: Training PyTorch models with fairness regularization

### 3. LagrangianFairnessTrainer
- Uses Lagrangian multipliers to enforce fairness constraints
- Alternates between optimizing model and updating constraint penalty
- Supports demographic_parity and equal_opportunity constraints
- **Use when**: You need explicit constraint satisfaction with dual variables

### 4. GroupFairnessCalibrator
- Post-processing calibration to improve fairness
- Separate calibrators per group (Platt scaling or isotonic regression)
- Ensures calibrated probabilities across groups
- **Use when**: You need to improve fairness of existing model predictions

### 5. Pareto Frontier Visualization
- `sweep_pareto()`: Trains models across range of fairness strengths
- `plot_pareto()`: Visualizes accuracy-fairness trade-offs
- Helps identify optimal operating points
- **Use when**: Exploring the fairness-accuracy trade-off space

### Usage Recommendations

- **For sklearn workflows**: Use `ReductionsWrapper` with your existing sklearn models
- **For PyTorch workflows**: Use `FairnessRegularizerLoss` for simple regularization or `LagrangianFairnessTrainer` for explicit constraints
- **For post-processing**: Use `GroupFairnessCalibrator` when you can't retrain but need to improve fairness
- **For exploration**: Use Pareto frontier tools to understand trade-offs before committing to a specific approach

### Next Steps

- Experiment with different constraint types and parameters
- Try combining multiple approaches (e.g., in-training + post-processing)
- Explore intersectional fairness by extending to multiple sensitive attributes
- See the documentation for more advanced usage patterns